## 线性SVM


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm

# 构建一个简单的二维二分类数据集（12个样本点）
# SVM 的核心思想：找到一个超平面（此处为直线），使得两类数据之间的间隔(margin)最大
data = np.array([
    [0.1, 0.7],
    [0.3, 0.6],
    [0.4, 0.1],
    [0.5, 0.4],
    [0.8, 0.04],
    [0.42, 0.6],
    [0.9, 0.4],
    [0.6, 0.5],
    [0.7, 0.2],
    [0.7, 0.67],
    [0.27, 0.8],
    [0.5, 0.72]
])# 建立数据集
label = [1] * 6 + [0] * 6 #前六个数据的label为1后六个为0

# 生成用于可视化的网格点，覆盖整个数据范围
x_min, x_max = data[:, 0].min() - 0.2, data[:, 0].max() + 0.2
y_min, y_max = data[:, 1].min() - 0.2, data[:, 1].max() + 0.2
# meshgrid 生成二维网格坐标矩阵，用于对平面上每个点进行预测
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.002),
                     np.arange(y_min, y_max, 0.002))
print(xx)

# 线性 SVM：使用线性核函数 K(x_i, x_j) = x_i^T * x_j
# C=0.001：很小的惩罚系数，允许更多误分类，产生更宽的间隔
# 目标函数：min 1/2 ||w||^2 + C * sum(xi_i)，即最大化间隔同时最小化误分类
model_linear = svm.SVC(kernel='linear', C = 0.001)
model_linear.fit(data, label) # 训练，求解支持向量和决策边界

# 对网格中每个点进行预测，生成分类区域图
Z = model_linear.predict(np.c_[xx.ravel(), yy.ravel()]) # 预测
Z = Z.reshape(xx.shape)
# 用等高线填充图(contourf)展示决策区域：不同颜色代表不同类别
plt.contourf(xx, yy, Z, cmap = plt.cm.ocean, alpha=0.6)
plt.scatter(data[:6, 0], data[:6, 1], marker='o', color='r', s=100, lw=3) # 类别1用红色圆点
plt.scatter(data[6:, 0], data[6:, 1], marker='x', color='k', s=100, lw=3) # 类别0用黑色叉号
plt.title('Linear SVM')
plt.show()

## 多项式SVM
对比不同最高次数的分类情况

In [ ]:
# 多项式 SVM：使用多项式核函数 K(x_i, x_j) = (gamma * x_i^T * x_j + coef0)^degree
# degree 控制多项式的最高次数，次数越高决策边界越复杂（非线性能力越强）
# 对比不同多项式次数下的分类效果
plt.figure(figsize=(16, 15))
 
for i, degree in enumerate([1, 3, 5, 7, 9, 12]):
    # C=0.0001：较小的惩罚系数，侧重于最大化间隔
    model_poly = svm.SVC(C=0.0001, kernel='poly', degree=degree) # 多项式核
    model_poly.fit(data, label)
    # 对网格点预测：ravel 将二维展平为一维，c_ 按列拼接为二维坐标
    # 预测结果 reshape 回网格形状用于绘制等高线图
    Z = model_poly.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.subplot(3, 2, i + 1)
    plt.subplots_adjust(wspace=0.4, hspace=0.4)
    plt.contourf(xx, yy, Z, cmap=plt.cm.ocean, alpha=0.6)
 
    plt.scatter(data[:6, 0], data[:6, 1], marker='o', color='r', s=100, lw=3)
    plt.scatter(data[6:, 0], data[6:, 1], marker='x', color='k', s=100, lw=3)
    plt.title('Poly SVM with $\degree=$' + str(degree))
plt.show()

## 高斯核SVM

对比不同gamma下的分类情况

In [ ]:
# 高斯核 SVM（RBF 核）：K(x_i, x_j) = exp(-gamma * ||x_i - x_j||^2)
# gamma 控制高斯函数的"宽度"：gamma 越大，每个样本的影响范围越小，决策边界越复杂
# gamma 越小，决策边界越平滑（可能欠拟合）
plt.figure(figsize=(16, 15))
 
for i, gamma in enumerate([1, 5, 15, 35, 45, 55]):
    # C=0.0001：较小的惩罚系数
    model_rbf = svm.SVC(kernel='rbf', gamma=gamma, C= 0.0001).fit(data, label)
 
    Z = model_rbf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.subplot(3, 2, i + 1)
    plt.subplots_adjust(wspace=0.4, hspace=0.4)
    plt.contourf(xx, yy, Z, cmap=plt.cm.ocean, alpha=0.6)
 
    plt.scatter(data[:6, 0], data[:6, 1], marker='o', color='r', s=100, lw=3)
    plt.scatter(data[6:, 0], data[6:, 1], marker='x', color='k', s=100, lw=3)
    plt.title('RBF SVM with $\gamma=$' + str(gamma))
plt.show()

## 测试不同SVM在Mnist数据集上的分类情况

In [ ]:
# --- MNIST 数据集上测试不同核函数的 SVM 分类效果 ---
from sklearn import svm
import numpy as np
from time import time
from sklearn.metrics import accuracy_score
from struct import unpack  # 用于解析二进制文件头
from sklearn.model_selection import GridSearchCV  # 网格搜索超参数调优

# 自定义函数：读取 MNIST IDX 格式的图像文件
# MNIST 图像文件格式：[magic number, 图像数量, 行数, 列数, 像素数据...]
def readimage(path):
    with open(path, 'rb') as f:
        magic, num, rows, cols = unpack('>4I', f.read(16))  # 读取4个无符号整数的文件头
        img = np.fromfile(f, dtype=np.uint8).reshape(num, 784)  # 每张图展平为784维向量
    return img

# 自定义函数：读取 MNIST IDX 格式的标签文件
def readlabel(path):
    with open(path, 'rb') as f:
        magic, num = unpack('>2I', f.read(8))  # 读取2个无符号整数的文件头
        lab = np.fromfile(f, dtype=np.uint8)
    return lab

# 加载 MNIST 训练集和测试集
train_data  = readimage("datasets/MNIST/raw/train-images-idx3-ubyte")
train_label = readlabel("datasets/MNIST/raw/train-labels-idx1-ubyte")
test_data   = readimage("datasets/MNIST/raw/t10k-images-idx3-ubyte")
test_label  = readlabel("datasets/MNIST/raw/t10k-labels-idx1-ubyte")
print(train_data.shape)
print(train_label.shape)

# 数据集较大，为节约时间只使用前2000张训练、200张测试
train_data=train_data[:2000]
train_label=train_label[:2000]
test_data=test_data[:200]
test_label=test_label[:200]

svc=svm.SVC()
# 使用高斯核（RBF 核）进行网格搜索
# GridSearchCV 自动遍历参数组合，通过交叉验证选择最优参数
parameters = {'kernel':['rbf'], 'C':[1]}
print("Train...")
clf=GridSearchCV(svc,parameters,n_jobs=-1)  # n_jobs=-1 使用全部 CPU 核心
start = time()
clf.fit(train_data, train_label)
end = time()
t = end - start
print('Train：%dmin%.3fsec' % (t//60, t - 60 * (t//60)))
prediction = clf.predict(test_data)  # 对测试数据进行预测
print("accuracy: ", accuracy_score(prediction, test_label))

# 手动计算测试集准确率（与 accuracy_score 结果一致）
accurate=[0]*10
sumall=[0]*10
i=0
j=0
while i<len(test_label):
    sumall[test_label[i]]+=1
    if prediction[i]==test_label[i]:
        j+=1
    i+=1
print("测试集准确率：",j/200)

In [ ]:
# 使用多项式核 SVM 在 MNIST 上进行分类
# 多项式核将数据映射到高维多项式特征空间，适合处理非线性可分问题
parameters = {'kernel':['poly'], 'C':[1]}
print("Train...")
clf=GridSearchCV(svc,parameters,n_jobs=-1)
start = time()
clf.fit(train_data, train_label)
end = time()
t = end - start
print('Train：%dmin%.3fsec' % (t//60, t - 60 * (t//60)))
prediction = clf.predict(test_data)
print("accuracy: ", accuracy_score(prediction, test_label))
accurate=[0]*10
sumall=[0]*10
i=0
j=0
while i<len(test_label):
    sumall[test_label[i]]+=1
    if prediction[i]==test_label[i]:
        j+=1
    i+=1
print("测试集准确率：",j/200)

In [ ]:
# 使用线性核 SVM 在 MNIST 上进行分类
# 线性核计算速度最快，但分类能力相对有限（假设数据线性可分）
# 对于 MNIST 这类图像数据，线性核也能获得不错的效果（因为像素特征已具有较好的线性可分性）
parameters = {'kernel':['linear'], 'C':[1]}
print("Train...")
clf=GridSearchCV(svc,parameters,n_jobs=-1)
start = time()
clf.fit(train_data, train_label)
end = time()
t = end - start
print('Train：%dmin%.3fsec' % (t//60, t - 60 * (t//60)))
prediction = clf.predict(test_data)
print("accuracy: ", accuracy_score(prediction, test_label))
accurate=[0]*10
sumall=[0]*10
i=0
j=0
while i<len(test_label):
    sumall[test_label[i]]+=1
    if prediction[i]==test_label[i]:
        j+=1
    i+=1
print("测试集准确率：",j/200)